# PROYECTO 06

## Monitor de Anomalías en Sensores IoT (Calidad del Aire)

**Nivel:** Intermedio

### OBJETIVO DE NEGOCIO
Supervisar la calidad del aire en distintas zonas de la ciudad, distinguiendo entre fallos técnicos de los sensores y alertas reales de contaminación.

### QUÉ DEBE HACER TU CÓDIGO (GUÍA DE TRABAJO)

* **Separar fallos de hardware:** Filtrar las lecturas donde el sensor falló (None o valores negativos como -999.0) para que no arruinen los cálculos estadísticos.

* **Calcular promedios por zona:** Obtener el nivel promedio de contaminación (pm25) de cada sensor utilizando solo las lecturas válidas.

* **Clasificar alertas ambientales:** Detectar las horas y zonas donde el valor válido superó el límite seguro (50.0), diferenciando entre alerta moderada y alerta crítica.

In [4]:
# Rango normal PM2.5: 0.0 a 50.0 ug/m3
lecturas_sensores = [
{"sensor_id": "S_NORTE", "hora": "08:00", "pm25": 22.4},
{"sensor_id": "S_NORTE", "hora": "09:00", "pm25": 28.1},
{"sensor_id": "S_NORTE", "hora": "10:00", "pm25": 185.0},
{"sensor_id": "S_NORTE", "hora": "11:00", "pm25": 31.0},
{"sensor_id": "S_SUR", "hora": "08:00", "pm25": 18.5},
{"sensor_id": "S_SUR", "hora": "09:00", "pm25": -999.0},
{"sensor_id": "S_SUR", "hora": "10:00", "pm25": None},
{"sensor_id": "S_SUR", "hora": "11:00", "pm25": 21.0},
{"sensor_id": "S_CENTRO", "hora": "08:00", "pm25": 42.0},
{"sensor_id": "S_CENTRO", "hora": "09:00", "pm25": 68.5},
{"sensor_id": "S_CENTRO", "hora": "10:00", "pm25": 74.2},
{"sensor_id": "S_CENTRO", "hora": "11:00", "pm25": 48.0}
]

# elegimos los limites, 150 se lo doy yo
LIMITE_SEGURO = 50.0
LIMITE_CRITICO = 150.0

# primero separamos las lecturas válidas y fallos


def separar_lecturas(lecturas):
    lecturas_validas = []
    fallos = []

    for lectura in lecturas:
        pm25 = lectura["pm25"]

        if pm25 is None or pm25 < 0:
            fallos.append(lectura)
        else:
            lecturas_validas.append(lectura)

    return lecturas_validas, fallos

# luego con otra función calculamos los promedios por sensor, sólo lecturas validas

def calcular_promedios_por_sensor(lecturas_validas):
    valores_por_sensor = {}

    # Agrupamos los valores válidos por sensor creando un nuevo diccionario
    for lectura in lecturas_validas:
        sensor = lectura["sensor_id"]
        pm25 = lectura["pm25"]
        # si el sensor no existe o no esta como clave creamos una nueva lista vacía para el
        if sensor not in valores_por_sensor:
            valores_por_sensor[sensor] = []

        valores_por_sensor[sensor].append(pm25)

    # Calculamos el promedio de cada lista
    promedios = {}

    for sensor, valores in valores_por_sensor.items():
        promedio = sum(valores) / len(valores)
        promedios[sensor] = promedio

    return promedios

# ahora con otra función detectamos las alertas ambientales

def detectar_alertas(lecturas_validas, limite_seguro, limite_critico):
    alertas = []

    for lectura in lecturas_validas:
        pm25 = lectura["pm25"]

        if pm25 > limite_critico:
            nivel = "CRÍTICA"
        elif pm25 > limite_seguro:
            nivel = "MODERADA"
        else:
            continue
        # creamos un nuevo diccionario con el nivel de alerta
        alerta = {
            "sensor_id": lectura["sensor_id"],
            "hora": lectura["hora"],
            "pm25": pm25,
            "nivel": nivel
        }

        alertas.append(alerta)

    return alertas

# ahora llamamos a las funciones

validas, fallos = separar_lecturas(lecturas_sensores)

promedios = calcular_promedios_por_sensor(validas)

alertas = detectar_alertas(
    validas,
    LIMITE_SEGURO,
    LIMITE_CRITICO
)

# y por último mostramos los resultados


print("=== PROMEDIOS POR SENSOR ===")

for sensor, promedio in promedios.items():
    print(f"{sensor}: {promedio:.2f} ug/m3")


print("\n=== FALLOS DE HARDWARE ===")

for fallo in fallos:
    print(
        f"{fallo['sensor_id']} a las {fallo['hora']}: "
        f"valor = {fallo['pm25']}"
    )


print("\n=== ALERTAS AMBIENTALES ===")

for alerta in alertas:
    print(
        f"{alerta['nivel']}: {alerta['sensor_id']} "
        f"a las {alerta['hora']} — {alerta['pm25']} ug/m3"
    )


=== PROMEDIOS POR SENSOR ===
S_NORTE: 66.62 ug/m3
S_SUR: 19.75 ug/m3
S_CENTRO: 58.17 ug/m3

=== FALLOS DE HARDWARE ===
S_SUR a las 09:00: valor = -999.0
S_SUR a las 10:00: valor = None

=== ALERTAS AMBIENTALES ===
CRÍTICA: S_NORTE a las 10:00 — 185.0 ug/m3
MODERADA: S_CENTRO a las 09:00 — 68.5 ug/m3
MODERADA: S_CENTRO a las 10:00 — 74.2 ug/m3
